In [29]:
! pip install PyPDF2
! pip install faiss-gpu
! pip install sentence-transformers
! pip install transformers

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [30]:
!pip install faiss-cpu # Added to resolve ModuleNotFoundError
import gradio as gr
import PyPDF2
import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [31]:
# Embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Question Answering model
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2" # Changed task to 'question-answering' and model to a compatible one
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
document_chunks = []
index = None

In [33]:
def extract_text_from_pdf(pdf_file):
    text = ""
    reader = PyPDF2.PdfReader(pdf_file)

    for page in reader.pages:
        text += page.extract_text()

    return text

In [34]:
def chunk_text(text, chunk_size=500):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    return chunks

In [35]:
def process_pdf(pdf_file):
    global document_chunks, index

    # Step 1: Extract text
    text = extract_text_from_pdf(pdf_file)

    # Step 2: Split into chunks
    document_chunks = chunk_text(text)

    # Step 3: Create embeddings
    embeddings = embedding_model.encode(document_chunks)
    embeddings = np.array(embeddings).astype("float32")

    # Step 4: Create FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return "PDF processed successfully!"

In [36]:
def answer_question(question):
    global document_chunks, index

    if index is None:
        return "Please upload a PDF first."

    # Step 1: Embed question
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    # Step 2: Search similar chunks
    D, I = index.search(question_embedding, k=5)

    # Step 3: Get context
    context = " ".join([document_chunks[i] for i in I[0]])

    # Step 4: Get answer
    result = qa_pipeline(question=question, context=context)

    return result["answer"]

In [37]:
with gr.Blocks() as app:
    gr.Markdown("# 📄 PDF Question Answering Bot")

    pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
    upload_button = gr.Button("Process PDF")
    status = gr.Textbox(label="Status")

    question_input = gr.Textbox(label="Ask a Question")
    answer_output = gr.Textbox(label="Answer")

    upload_button.click(process_pdf, inputs=pdf_input, outputs=status)
    question_input.submit(answer_question, inputs=question_input, outputs=answer_output)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://120fabeaba5ddd6796.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
